# OmniDev: Autonomous Polyglot Technical Mentor and Execution Workbench
## Week 2 Capstone Project — Multi-Modal Gradio Blocks Studio with Tool Calling, Live Python Sandbox, SQLite Knowledge Vault, and Bidirectional Voice

### Architectural Overview
This capstone project upgrades the Week 1 Technical Question/Answerer into an interactive developer workbench built on `gr.Blocks`. It integrates six core LLM engineering patterns:

1. **Multi-Provider Model Routing:** Dynamically switches between Google Gemini models, OpenRouter endpoints, and OpenAI models from the UI.
2. **Dynamic System Prompting & Expertise Modulation:** Adjusts pedagogical depth (First-Principles Intuition, Production Staff Engineer, Socratic Debugger, Executive Summary) and injects live code from the workbench editor.
3. **Autonomous Tool-Calling Relay:** Equips the LLM with three functional developer tools:
   - `execute_python_sandbox`: Executes Python code snippets and captures `stdout` to verify logic and mathematical outputs.
   - `save_technical_note`: Saves structured technical notes and code snippets into a persistent SQLite database (`engineering_vault.db`).
   - `search_engineering_vault`: Queries saved engineering notes from the SQLite database.
4. **Real-Time Token Streaming:** Streams technical explanations token-by-token into the chat interface after resolving tool calls.
5. **Bidirectional Voice Interface:** Captures spoken microphone questions via multimodal audio transcription and speaks concise audio summaries using neural Text-to-Speech (`en-US-ChristopherNeural`).
6. **Live Code Extraction & Telemetry Inspector:** Automatically extracts generated code blocks into a syntax-highlighted `gr.Code` editor and logs every JSON tool invocation in a live `gr.JSON` telemetry view.

## 1. Imports and Asynchronous Loop Configuration

### How This Works
- Standard libraries (`os`, `io`, `re`, `json`, `base64`, `sqlite3`, `contextlib`, `traceback`) handle JSON tool parsing, SQLite database persistence, and safe `stdout` redirection for the Python execution sandbox.
- `nest_asyncio.apply()` patches Jupyter's running event loop so asynchronous neural Text-to-Speech (`edge_tts`) can execute inside standard synchronous Gradio callbacks without raising runtime loop conflicts.

In [ ]:
import os
import io
import re
import json
import base64
import sqlite3
import asyncio
import contextlib
import traceback
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor

import requests
import edge_tts
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)

## 2. Multi-Provider Client Initialization and Model Registry

### How This Works
To allow seamless switching between different model families in the Gradio UI, we initialize OpenAI-compatible client wrappers for Google Gemini, OpenRouter, and standard OpenAI.

The `MODEL_REGISTRY` dictionary maps each user-facing dropdown label to its corresponding `(client, model_id, supports_tools)` configuration. When the user switches models in the UI, the backend dynamically routes the request to the selected provider endpoint.

In [ ]:
google_api_key = os.getenv("GOOGLE_API_KEY")
openai_api_key = os.getenv("OPENAI_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

GENINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1/"

gemini_client = OpenAI(api_key=google_api_key or "unconfigured", base_url=GENINI_BASE_URL)
openrouter_client = OpenAI(api_key=openrouter_api_key or "unconfigured", base_url=OPENROUTER_BASE_URL)
openai_client = OpenAI(api_key=openai_api_key or "unconfigured")

MODEL_REGISTRY = {
    "Gemini 3.1 Flash Lite (Free Tier - Fast & Tool Ready)": {
        "client": gemini_client,
        "model": "gemini-3.1-flash-lite",
        "supports_tools": True
    },
    "Gemini 2.5 Flash (Free Tier - Deep Reasoning)": {
        "client": gemini_client,
        "model": "gemini-2.5-flash",
        "supports_tools": True
    },
    "OpenRouter Auto Free (openrouter/free)": {
        "client": openrouter_client,
        "model": "openrouter/free",
        "supports_tools": False
    },
    "OpenAI GPT-4.1 Mini (Standard Paid API)": {
        "client": openai_client,
        "model": "gpt-4.1-mini",
        "supports_tools": True
    }
}

DEFAULT_MODEL_KEY = "Gemini 3.1 Flash Lite (Free Tier - Fast & Tool Ready)"
print(f"Configured {len(MODEL_REGISTRY)} model endpoints. Default: {DEFAULT_MODEL_KEY}")


## 3. Persistent SQLite Engineering Vault

### How This Works
A technical mentor becomes much more useful when it can save and retrieve architectural notes, debugging patterns, and code snippets.
- `init_vault_db()` creates a local SQLite database file named `engineering_vault.db` containing the `notes` table (`id`, `topic`, `category`, `summary`, `timestamp`).
- `fetch_vault_table()` queries the database and formats all stored records as a 2D list to keep our Gradio `gr.Dataframe` component synchronized in real time.

In [ ]:
DB_PATH = "engineering_vault.db"

def init_vault_db():
    with sqlite3.connect(DB_PATH) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            CREATE TABLE IF NOT EXISTS notes (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                topic TEXT NOT NULL,
                category TEXT NOT NULL,
                summary TEXT NOT NULL,
                timestamp TEXT NOT NULL
            )
        """)
        
        # Seed with one starter record if table is empty
        
        cursor.execute("SELECT COUNT(*) FROM notes")
        if cursor.fetchone()[0] == 0:
            cursor.execute(
                "INSERT INTO notes (topic, category, summary, timestamp) VALUES (?, ?, ?, ?)",
                (
                    "Python Generators & Yield",
                    "Python Core",
                    "Use 'yield' to stream tokens lazily over HTTP without buffering the full response in memory.",
                    datetime.now().strftime("%Y-%m-%d %H:%M")
                )
            )
        conn.commit()
        

def fetch_vault_table():
    with sqlite3.connect(DB_PATH) as conn:
        cursor = conn.cursor()
        cursor.execute("SELECT id, topic, category, summary, timestamp FROM notes ORDER BY id DESC")
        rows = cursor.fetchall()
    return [[r[0], r[1], r[2], r[3], r[4]] for r in rows]


init_vault_db()
print(f"SQLite Engineering Vault initialized. Current records: {len(fetch_vault_table())}")
        

## 4. Autonomous Developer Tools and Function Schemas

### How This Works
We equip the assistant with three distinct Python tools:
1. **`execute_python_sandbox(code)`**: Executes pure Python calculations or algorithm demonstrations inside a restricted namespace using `contextlib.redirect_stdout` and returns the exact printed output or traceback to the LLM.
2. **`save_technical_note(topic, category, summary)`**: Inserts a new technical takeaway into `engineering_vault.db`.
3. **`search_engineering_vault(query)`**: Performs a SQL `LIKE` search across saved notes in `engineering_vault.db` so the LLM can reference prior notes.

Each function is paired with a strict JSON Schema inside the `TOOLS_MENU` list and routed through `execute_tool_calls()`.

In [ ]:
# --- Tool 1: Safe Python Execution Sandbox ---
def execute_python_sandbox(code: str) -> str:
    print("TOOL CALLED: execute_python_sandbox", flush=True)
    buffer = io.StringIO()
    safe_globals = {
        "__builtins__": {
            "print": print, "range": range, "len": len, "int": int, "float": float,
            "str": str, "list": list, "dict": dict, "set": set, "tuple": tuple,
            "sum": sum, "min": min, "max": max, "abs": abs, "round": round,
            "sorted": sorted, "enumerate": enumerate, "zip": zip, "map": map,
            "filter": filter, "any": any, "all": all, "isinstance": isinstance
        }
    }
    local_env = {}
    try:
        with contextlib.redirect_stdout(buffer):
            exec(code, safe_globals, local_env)
        output = buffer.getvalue().strip()
        if not output and "result" in local_env:
            output = f"result = {local_env['result']}"
        return f"Sandbox Execution Succeeded:\n{output or '(No stdout printed; use print() to see output)'}"
    except Exception:
        return f"Sandbox Execution Error:\n{traceback.format_exc(limit=3)}"

In [ ]:
# --- Tool 2: Save Technical Note to SQLite ---
def save_technical_note(topic: str, category: str, summary: str) -> str:
    print(f"TOOL CALLED: save_technical_note ({topic})", flush=True)
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
    with sqlite3.connect(DB_PATH) as conn:
        cursor = conn.cursor()
        cursor.execute(
            "INSERT INTO notes (topic, category, summary, timestamp) VALUES (?, ?, ?, ?)",
            (topic.strip(), category.strip(), summary.strip(), timestamp)
        )
        conn.commit()
    return f"Successfully saved note '{topic}' under category '{category}' in the SQLite Engineering Vault."

In [ ]:
# --- Tool 3: Search SQLite Engineering Vault ---
def search_engineering_vault(query: str) -> str:
    print(f"TOOL CALLED: search_engineering_vault ({query})", flush=True)
    with sqlite3.connect(DB_PATH) as conn:
        cursor = conn.cursor()
        pattern = f"%{query.strip()}%"
        cursor.execute(
            "SELECT id, topic, category, summary, timestamp FROM notes WHERE topic LIKE ? OR category LIKE ? OR summary LIKE ? ORDER BY id DESC LIMIT 5",
            (pattern, pattern, pattern)
        )
        rows = cursor.fetchall()
    if not rows:
        return f"No saved notes found matching '{query}'."
    formatted = [f"[{r[0]}] {r[1]} ({r[2]} - {r[4]}): {r[3]}" for r in rows]
    return "Matching Engineering Vault Notes:\n" + "\n".join(formatted)

In [ ]:
# --- JSON Schemas for LLM Tool Calling ---
sandbox_schema = {
    "name": "execute_python_sandbox",
    "description": "Execute a self-contained Python code snippet (using print statements) to verify algorithms, data structures, or math calculations before answering.",
    "parameters": {
        "type": "object",
        "properties": {
            "code": {
                "type": "string",
                "description": "Valid Python code snippet using print() to display output."
            }
        },
        "required": ["code"],
        "additionalProperties": False
    }
}

In [ ]:
save_note_schema = {
    "name": "save_technical_note",
    "description": "Save a key technical concept, architectural pattern, or debugging takeaway into the persistent SQLite Engineering Vault when requested by the user or when summarizing an important lesson.",
    "parameters": {
        "type": "object",
        "properties": {
            "topic": {"type": "string", "description": "Short title of the technical concept"},
            "category": {"type": "string", "description": "Domain category (e.g., Python, LLM Engineering, SQL, System Design)"},
            "summary": {"type": "string", "description": "Concise 1-2 sentence technical takeaway"}
        },
        "required": ["topic", "category", "summary"],
        "additionalProperties": False
    }
}

In [ ]:
search_vault_schema = {
    "name": "search_engineering_vault",
    "description": "Search the SQLite Engineering Vault for previously saved technical notes, cheat sheets, or architectural summaries.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Keyword or topic to search for in the database"}
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [ ]:
TOOLS_MENU = [
    {"type": "function", "function": sandbox_schema},
    {"type": "function", "function": save_note_schema},
    {"type": "function", "function": search_vault_schema}
]

In [ ]:
def execute_tool_calls(assistant_message):
    tool_responses = []
    telemetry_records = []
    latest_sandbox_output = None

    for tool_call in assistant_message.tool_calls:
        fn_name = tool_call.function.name
        try:
            args = json.loads(tool_call.function.arguments)
        except Exception:
            args = {}

        if fn_name == "execute_python_sandbox":
            code_str = args.get("code", "")
            result_str = execute_python_sandbox(code_str)
            latest_sandbox_output = result_str
        elif fn_name == "save_technical_note":
            result_str = save_technical_note(
                args.get("topic", "Untitled"),
                args.get("category", "General"),
                args.get("summary", "")
            )
        elif fn_name == "search_engineering_vault":
            result_str = search_engineering_vault(args.get("query", ""))
        else:
            result_str = f"Unrecognized tool: {fn_name}"

        tool_responses.append({
            "role": "tool",
            "content": result_str,
            "tool_call_id": tool_call.id
        })
        telemetry_records.append({
            "id": tool_call.id,
            "tool": fn_name,
            "arguments": args,
            "output": result_str
        })

    return tool_responses, telemetry_records, latest_sandbox_output

## 5. Bidirectional Audio Pipeline (Voice Input & Bold Male Voice Output)

### How This Works
To fulfill the bonus multimodal challenge ("add audio input so you can talk to it, and have it respond with audio"), we implement two zero-cost functions:
1. **`transcribe_voice_input(audio_filepath)`**: Reads the recorded microphone WAV/MP3 file, encodes it in Base64, and sends it to Google Gemini's multimodal endpoint (`gemini-3.1-flash-lite`) to transcribe spoken technical questions accurately into text without requiring local `ffmpeg` binaries.
2. **`synthesize_male_voice(text, enabled)`**: Uses `edge_tts` with the `en-US-ChristopherNeural` voice preset to strip out raw code blocks and speak a concise, authoritative audio briefing of the explanation.

In [ ]:
def transcribe_voice_input(audio_filepath):
    """
    Transcribes microphone audio files using Gemini's native multimodal REST API.
    """
    if not audio_filepath or not google_api_key:
        return ""
    try:
        with open(audio_filepath, "rb") as f:
            audio_bytes = base64.b64encode(f.read()).decode("utf-8")

        url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-3.1-flash-lite:generateContent?key={google_api_key}"
        payload = {
            "contents": [{
                "parts": [
                    {"text": "Transcribe this spoken technical question accurately into plain text. Return ONLY the transcribed text, nothing else."},
                    {"inline_data": {"mime_type": "audio/wav", "data": audio_bytes}}
                ]
            }]
        }
        response = requests.post(url, json=payload, timeout=15)
        if response.status_code == 200:
            data = response.json()
            return data["candidates"][0]["content"]["parts"][0]["text"].strip()
    except Exception as err:
        print(f"Audio transcription error: {err}")
    return ""

def synthesize_male_voice(markdown_text: str, voice_enabled: bool = True):
    """
    Synthesizes a deep, bold male voice summary using Microsoft Neural TTS (en-US-ChristopherNeural).
    Runs in an isolated worker thread so Jupyter and Gradio event loops never conflict.
    """
    if not voice_enabled or not markdown_text:
        return None

    # Strip fenced code blocks and markdown formatting so the spoken summary sounds natural
    spoken_text = re.sub(r"```[\s\S]*?```", " See the code workbench for the implementation details. ", markdown_text)
    spoken_text = re.sub(r"[#*`_~>]", "", spoken_text)
    spoken_text = " ".join(spoken_text.split())[:420]

    async def _run_tts():
        communicate = edge_tts.Communicate(spoken_text, voice="en-US-ChristopherNeural")
        audio_data = bytearray()
        async for chunk in communicate.stream():
            if chunk["type"] == "audio":
                audio_data.extend(chunk["data"])
        return bytes(audio_data)

    try:
        with ThreadPoolExecutor(max_workers=1) as executor:
            return executor.submit(asyncio.run, _run_tts()).result()
    except Exception as err:
        print(f"TTS generation error: {err}")
        return None

## 6. Dynamic System Prompting and Streaming Tool-Call Orchestrator

### How This Works
1. **`build_mentor_system_prompt(domain, mode, workbench_code)`**: Dynamically constructs the system prompt based on the selected Engineering Domain, Pedagogical Depth Mode, and any code currently present in the user's interactive Code Workbench.
2. **`extract_python_code(markdown_text)`**: Parses the assistant's Markdown response to extract any Python code block (` ```python ... ``` `) and automatically loads it into the right-hand Code Workbench.
3. **`stream_mentor_response(...)`**: Implements a two-stage hybrid generator:
   - **Stage 1 (Tool Resolution):** If the selected model supports tools, it resolves any `execute_python_sandbox`, `save_technical_note`, or `search_engineering_vault` calls in a `while` loop while yielding live status updates to the UI.
   - **Stage 2 (Token Streaming):** Streams the final explanation token-by-token using `stream=True` and `yield`, then synthesizes the voice audio and updates the SQLite table and code workbench on the final yield.

In [ ]:
import time

def build_mentor_system_prompt(domain: str, mode: str, workbench_code: str) -> str:
    mode_instructions = {
        "Staff Engineer (Production Best Practices)": "Provide rigorous, production-grade explanations focusing on architecture, edge cases, time/space complexity, and clean modular code.",
        "First-Principles Intuition (Step-by-Step)": "Break complex ideas down from first principles using clear mental models, step-by-step trace walkthroughs, and annotated examples.",
        "Socratic Debugger (Root-Cause Analysis)": "Diagnose bugs systematically, explain the exact root cause, verify fixes using the Python sandbox tool when appropriate, and show the corrected code.",
        "Concise Executive Cheat-Sheet": "Deliver high-density bullet points, syntax tables, and minimal runnable code snippets without fluff."
    }

    prompt = f"""You are OmniDev, a Principal AI & Software Engineering Technical Mentor specializing in {domain}.
Pedagogical Style: {mode}
Style Directive: {mode_instructions.get(mode, mode_instructions["Staff Engineer (Production Best Practices)"])}

Tool Usage Guidelines:
- Use `execute_python_sandbox` whenever verifying Python output, demonstrating an algorithm, or testing a mathematical/code question.
- Use `save_technical_note` whenever the user asks to save/bookmark a note, or when recording a critical architectural rule to the Engineering Vault.
- Use `search_engineering_vault` whenever the user asks what notes are saved in the vault or references past notes.
- Format all explanations in clean Markdown. Wrap Python examples in ```python ... ``` blocks.
"""
    if workbench_code and workbench_code.strip():
        prompt += f"\n--- USER'S CURRENT WORKBENCH CODE ---\n```python\n{workbench_code.strip()}\n```\nIf relevant to the user's question, analyze, debug, or improve this workbench code.\n"
    return prompt

def extract_python_code(text: str):
    matches = re.findall(r"```python\s*([\s\S]*?)```", text)
    if matches:
        return matches[-1].strip()
    return None

def stream_mentor_response(
    chat_history,
    model_key,
    domain,
    mode,
    workbench_code,
    voice_enabled,
    current_telemetry,
    current_sandbox_out
):
    if not chat_history or chat_history[-1]["role"] != "user":
        yield chat_history, None, workbench_code, current_sandbox_out, fetch_vault_table(), current_telemetry
        return

    model_cfg = MODEL_REGISTRY.get(model_key, MODEL_REGISTRY[DEFAULT_MODEL_KEY])
    active_client = model_cfg["client"]
    active_model = model_cfg["model"]
    supports_tools = model_cfg["supports_tools"]

    sys_prompt = build_mentor_system_prompt(domain, mode, workbench_code)
    clean_history = [{"role": h["role"], "content": h["content"]} for h in chat_history]
    messages = [{"role": "system", "content": sys_prompt}] + clean_history

    telemetry_list = list(current_telemetry) if isinstance(current_telemetry, list) else []
    sandbox_out = current_sandbox_out or "No sandbox execution in this turn."

    # Add placeholder assistant message for streaming updates
    chat_history = chat_history + [{"role": "assistant", "content": "Thinking..."}]
    yield chat_history, None, workbench_code, sandbox_out, fetch_vault_table(), telemetry_list

    max_retries = 3

    try:
        # Stage 1: Tool Resolution Loop with 429 Rate-Limit Protection
        if supports_tools:
            probe = None
            for attempt in range(max_retries):
                try:
                    probe = active_client.chat.completions.create(
                        model=active_model,
                        messages=messages,
                        tools=TOOLS_MENU
                    )
                    break
                except Exception as api_err:
                    if "429" in str(api_err) and attempt < max_retries - 1:
                        wait_time = (attempt + 1) * 2
                        chat_history[-1]["content"] = f"Rate limit reached (429). Retrying tool probe in {wait_time}s..."
                        yield chat_history, None, workbench_code, sandbox_out, fetch_vault_table(), telemetry_list
                        time.sleep(wait_time)
                    else:
                        raise api_err

            while probe and probe.choices[0].finish_reason == "tool_calls":
                assistant_msg = probe.choices[0].message
                tool_names = ", ".join([tc.function.name for tc in assistant_msg.tool_calls])
                chat_history[-1]["content"] = f"Executing developer tools: `{tool_names}`..."
                yield chat_history, None, workbench_code, sandbox_out, fetch_vault_table(), telemetry_list

                tool_outputs, new_telemetry, latest_sb = execute_tool_calls(assistant_msg)
                telemetry_list.extend(new_telemetry)
                if latest_sb:
                    sandbox_out = latest_sb

                messages.append(assistant_msg)
                messages.extend(tool_outputs)

                # Retry block for follow-up tool calls
                for attempt in range(max_retries):
                    try:
                        probe = active_client.chat.completions.create(
                            model=active_model,
                            messages=messages,
                            tools=TOOLS_MENU
                        )
                        break
                    except Exception as api_err:
                        if "429" in str(api_err) and attempt < max_retries - 1:
                            time.sleep((attempt + 1) * 2)
                        else:
                            raise api_err

        # Stage 2: Real-Time Token Streaming Call with 429 Rate-Limit Protection
        stream = None
        for attempt in range(max_retries):
            try:
                stream = active_client.chat.completions.create(
                    model=active_model,
                    messages=messages,
                    stream=True
                )
                break
            except Exception as api_err:
                if "429" in str(api_err) and attempt < max_retries - 1:
                    wait_time = (attempt + 1) * 2
                    chat_history[-1]["content"] = f"Rate limit reached (429). Retrying generation in {wait_time}s..."
                    yield chat_history, None, workbench_code, sandbox_out, fetch_vault_table(), telemetry_list
                    time.sleep(wait_time)
                else:
                    raise api_err

        accumulated = ""
        for chunk in stream:
            if chunk.choices and chunk.choices[0].delta.content:
                accumulated += chunk.choices[0].delta.content
                chat_history[-1]["content"] = accumulated
                yield chat_history, None, workbench_code, sandbox_out, fetch_vault_table(), telemetry_list

        # Extract any newly generated Python code to populate the live Code Workbench
        extracted_code = extract_python_code(accumulated)
        updated_workbench = extracted_code if extracted_code else workbench_code

        # Generate bold male voice audio once streaming finishes
        audio_bytes = synthesize_male_voice(accumulated, voice_enabled=voice_enabled)

        yield chat_history, audio_bytes, updated_workbench, sandbox_out, fetch_vault_table(), telemetry_list

    except Exception as err:
        chat_history[-1]["content"] = f"**Execution Error ({active_model}):** {str(err)}"
        yield chat_history, None, workbench_code, sandbox_out, fetch_vault_table(), telemetry_list

## 7. Gradio Blocks User Interface and Event Chaining

### How This Works
We construct a custom monochrome dark interface using `gr.Blocks` and custom CSS:
- **Top Control Ribbon:** Lets the user switch between LLM providers/models, select the Technical Domain, choose the Pedagogical Mode, and toggle the Neural Male Voice output.
- **Left Column (Conversational Debugger & Voice Input):**
  - `gr.Chatbot` with `type="messages"` renders streamed Markdown and syntax-highlighted code.
  - `gr.Audio(sources=["microphone", "upload"])` allows spoken questions that automatically transcribe into the input box.
  - Clickable `gr.Examples` test tool calling, sandbox verification, and SQLite vault persistence.
- **Right Column (3-Tab Engineering Workbench):**
  - **Tab 1 (`Interactive Code Sandbox`):** A live `gr.Code(language="python")` editor where users can edit code, run it directly in the local Python sandbox with one click, or let the AI inspect it.
  - **Tab 2 (`SQLite Engineering Vault`):** A live `gr.Dataframe` displaying saved technical notes in `engineering_vault.db`.
  - **Tab 3 (`Tool Call Telemetry`):** A `gr.JSON` inspector showing every tool call's arguments and database/sandbox response.

In [ ]:
CUSTOM_CSS = """
/* Global Container & Background */
.gradio-container {
    max-width: 1600px !important;
    background-color: #07080b !important;
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important;
    color: #f3f4f6 !important;
}

/* Studio Header Styling */
.studio-header {
    background: linear-gradient(135deg, #0e1017 0%, #161922 100%);
    border: 1px solid #1f2533;
    padding: 22px 30px;
    border-radius: 14px;
    margin-bottom: 16px;
    box-shadow: 0 10px 30px rgba(0, 0, 0, 0.5);
}

/* Control Panel Ribbon */
.control-panel {
    background: #0e1017 !important;
    border: 1px solid #1f2533 !important;
    border-radius: 12px !important;
    padding: 14px 18px !important;
    margin-bottom: 16px !important;
    box-shadow: 0 4px 20px rgba(0, 0, 0, 0.3);
}

/* Glassmorphic Workspace Cards */
.workspace-card {
    background: #0e1017 !important;
    border: 1px solid #1f2533 !important;
    border-radius: 14px !important;
    padding: 18px !important;
    box-shadow: 0 8px 32px rgba(0, 0, 0, 0.4);
}

/* Custom Primary Button Styling (Vercel / OpenAI Glow Effect) */
button.primary {
    background: linear-gradient(135deg, #3b82f6 0%, #1d4ed8 100%) !important;
    border: none !important;
    color: white !important;
    font-weight: 600 !important;
    border-radius: 8px !important;
    transition: all 0.2s ease-in-out !important;
    box-shadow: 0 4px 16px rgba(37, 99, 235, 0.35) !important;
}
button.primary:hover {
    background: linear-gradient(135deg, #60a5fa 0%, #2563eb 100%) !important;
    box-shadow: 0 6px 22px rgba(37, 99, 235, 0.55) !important;
    transform: translateY(-1px);
}

/* Secondary Button Styling */
button.secondary {
    background: #171b26 !important;
    border: 1px solid #283044 !important;
    color: #cbd5e1 !important;
    border-radius: 8px !important;
    transition: all 0.2s ease-in-out !important;
}
button.secondary:hover {
    background: #212838 !important;
    color: #ffffff !important;
    border-color: #3b82f6 !important;
}

/* Form Inputs, Dropdowns and Textboxes */
.gr-input, .gr-box, textarea, select {
    background-color: #0b0d12 !important;
    border: 1px solid #1f2533 !important;
    color: #f8fafc !important;
    border-radius: 8px !important;
}
.gr-input:focus, textarea:focus {
    border-color: #3b82f6 !important;
    box-shadow: 0 0 0 2px rgba(59, 130, 246, 0.25) !important;
}

/* Chatbot Window Styling */
.gr-chatbot {
    background-color: #0b0d12 !important;
    border: 1px solid #1f2533 !important;
    border-radius: 10px !important;
}

/* Code Editor & Dataframe Panels */
.gr-code, .gr-dataframe {
    border-radius: 10px !important;
    border: 1px solid #1f2533 !important;
}
"""

def append_user_message(user_text, chat_history):
    if not user_text or not user_text.strip():
        return "", chat_history
    return "", chat_history + [{"role": "user", "content": user_text.strip()}]

def manual_run_workbench(code_str):
    if not code_str or not code_str.strip():
        return "No code in workbench to execute."
    return execute_python_sandbox(code_str)

with gr.Blocks(
    theme=gr.themes.Monochrome(
        primary_hue="blue",
        secondary_hue="slate",
        radius_size=gr.themes.sizes.radius_sm,
        font=[gr.themes.GoogleFont("Inter"), "sans-serif"],
        font_mono=[gr.themes.GoogleFont("JetBrains Mono"), "monospace"]
    ),
    css=CUSTOM_CSS,
    title="OmniDev // Enterprise AI Technical Mentor"
) as omnidev_studio:

    # MNC-Style Header Banner with Live Pulse Badge
    gr.HTML("""
    <div class="studio-header">
        <div style="display: flex; justify-content: space-between; align-items: center;">
            <div>
                <span style="background: rgba(59, 130, 246, 0.15); color: #60a5fa; padding: 4px 10px; border-radius: 20px; font-size: 0.75rem; font-weight: 600; text-transform: uppercase; letter-spacing: 0.05em;">Enterprise Edition</span>
                <h1 style="margin: 8px 0 4px 0; color: #ffffff; font-size: 1.6rem; font-weight: 700; letter-spacing: -0.025em;">OmniDev // Autonomous Technical Mentor & Execution Workbench</h1>
                <p style="margin: 0; color: #94a3b8; font-size: 0.9rem;">
                    Multi-Model Architecture | Isolated Python Sandbox | Persistent SQLite Knowledge Vault | Neural Audio Briefings
                </p>
            </div>
            <div style="display: flex; align-items: center; gap: 8px; background: #0b0d12; padding: 8px 14px; border-radius: 8px; border: 1px solid #1f2533;">
                <div style="width: 8px; height: 8px; background: #22c55e; border-radius: 50%; box-shadow: 0 0 10px #22c55e;"></div>
                <span style="font-size: 0.82rem; color: #cbd5e1; font-weight: 500;">System Online</span>
            </div>
        </div>
    </div>
    """)

    # Top Configuration Ribbon (Card Wrapper)
    with gr.Row(elem_classes=["control-panel"]):
        model_dropdown = gr.Dropdown(
            choices=list(MODEL_REGISTRY.keys()),
            value=DEFAULT_MODEL_KEY,
            label="Active LLM Engine",
            scale=3
        )
        domain_dropdown = gr.Dropdown(
            choices=[
                "Python & Backend Architecture",
                "LLM Engineering & Agentic Systems",
                "Data Structures & Algorithms",
                "Full-Stack React & Node.js",
                "SQL & Database Optimization"
            ],
            value="LLM Engineering & Agentic Systems",
            label="Technical Domain",
            scale=3
        )
        mode_dropdown = gr.Dropdown(
            choices=[
                "Staff Engineer (Production Best Practices)",
                "First-Principles Intuition (Step-by-Step)",
                "Socratic Debugger (Root-Cause Analysis)",
                "Concise Executive Cheat-Sheet"
            ],
            value="Staff Engineer (Production Best Practices)",
            label="Pedagogical Mode",
            scale=3
        )
        voice_toggle = gr.Checkbox(
            value=True,
            label="Enable Neural Male Voice (TTS)",
            scale=2
        )

    # Main Workspace Split (Left & Right Cards)
    with gr.Row():
        # Left Column: Conversational Debugger Card
        with gr.Column(scale=6, elem_classes=["workspace-card"]):
            gr.Markdown("### <span style='color: #60a5fa;'>💬</span> Interactive Mentor Session")
            chatbot = gr.Chatbot(
                label="",
                type="messages",
                height=420
            )
            question_box = gr.Textbox(
                label="",
                placeholder="Ask a technical question, command a tool, or record via microphone below...",
                lines=2
            )
            with gr.Row():
                submit_btn = gr.Button("Ask OmniDev (Stream + Tools)", variant="primary", scale=3)
                clear_btn = gr.Button("Reset Session", variant="secondary", scale=1)

            with gr.Row():
                mic_input = gr.Audio(
                    sources=["microphone", "upload"],
                    type="filepath",
                    label="Voice Question Input (Auto-Transcribe)"
                )
                voice_output = gr.Audio(
                    label="Executive Voice Briefing (ChristopherNeural)",
                    autoplay=True
                )

            gr.Examples(
                examples=[
                    "Write a Python function to compute the 15th Fibonacci number, run it in your Python sandbox tool to verify the exact output, and save a summary note to the SQLite vault.",
                    "Explain why my recursive function in the Code Workbench on the right is slow, fix it with memoization, and verify the speedup in the sandbox.",
                    "Search my SQLite Engineering Vault and show me all technical notes currently saved.",
                    "How does OpenAI/Gemini tool calling work under the hood with finish_reason == 'tool_calls'?"
                ],
                inputs=question_box,
                label="Quick Test Scenarios"
            )

        # Right Column: Engineering Workbench Card
        with gr.Column(scale=5, elem_classes=["workspace-card"]):
            with gr.Tabs():
                with gr.TabItem("⚡ Interactive Code Workbench"):
                    gr.Markdown("<p style='font-size: 0.85rem; color: #94a3b8; margin-top: 4px;'>Shared execution context with the AI mentor.</p>")
                    code_workbench = gr.Code(
                        value="# Paste code here for OmniDev to analyze, or let OmniDev generate code into this editor\ndef fib_slow(n):\n    if n <= 1:\n        return n\n    return fib_slow(n - 1) + fib_slow(n - 2)\n\nprint('fib_slow(10) =', fib_slow(10))",
                        language="python",
                        label="",
                        lines=13
                    )
                    run_code_btn = gr.Button("Run Workbench Code in Local Sandbox", variant="secondary")
                    sandbox_output_box = gr.Textbox(
                        label="Sandbox Execution Output",
                        value="Ready.",
                        lines=5,
                        interactive=False
                    )

                with gr.TabItem("🗄️ SQLite Engineering Vault"):
                    gr.Markdown("<p style='font-size: 0.85rem; color: #94a3b8; margin-top: 4px;'>Persistent architectural takeaways stored in engineering_vault.db</p>")
                    vault_dataframe = gr.Dataframe(
                        headers=["ID", "Topic", "Category", "Technical Summary", "Timestamp"],
                        value=fetch_vault_table(),
                        interactive=False,
                        wrap=True,
                        label=""
                    )
                    refresh_vault_btn = gr.Button("Refresh Vault Records", size="sm", variant="secondary")

                with gr.TabItem("📊 Tool Call Telemetry"):
                    gr.Markdown("<p style='font-size: 0.85rem; color: #94a3b8; margin-top: 4px;'>Inspect raw JSON payloads of executed LLM tool calls.</p>")
                    telemetry_inspector = gr.JSON(
                        value=[],
                        label=""
                    )

    # --- Event Wiring ---
    question_box.submit(
        fn=append_user_message,
        inputs=[question_box, chatbot],
        outputs=[question_box, chatbot]
    ).then(
        fn=stream_mentor_response,
        inputs=[
            chatbot, model_dropdown, domain_dropdown, mode_dropdown,
            code_workbench, voice_toggle, telemetry_inspector, sandbox_output_box
        ],
        outputs=[
            chatbot, voice_output, code_workbench,
            sandbox_output_box, vault_dataframe, telemetry_inspector
        ]
    )

    submit_btn.click(
        fn=append_user_message,
        inputs=[question_box, chatbot],
        outputs=[question_box, chatbot]
    ).then(
        fn=stream_mentor_response,
        inputs=[
            chatbot, model_dropdown, domain_dropdown, mode_dropdown,
            code_workbench, voice_toggle, telemetry_inspector, sandbox_output_box
        ],
        outputs=[
            chatbot, voice_output, code_workbench,
            sandbox_output_box, vault_dataframe, telemetry_inspector
        ]
    )

    mic_input.stop_recording(
        fn=transcribe_voice_input,
        inputs=[mic_input],
        outputs=[question_box]
    )
    mic_input.upload(
        fn=transcribe_voice_input,
        inputs=[mic_input],
        outputs=[question_box]
    )

    run_code_btn.click(
        fn=manual_run_workbench,
        inputs=[code_workbench],
        outputs=[sandbox_output_box]
    )

    refresh_vault_btn.click(fn=fetch_vault_table, inputs=[], outputs=[vault_dataframe])

    clear_btn.click(
        fn=lambda: ("", [], None, "Ready.", fetch_vault_table(), []),
        inputs=[],
        outputs=[question_box, chatbot, voice_output, sandbox_output_box, vault_dataframe, telemetry_inspector]
    )

omnidev_studio.launch(inbrowser=True)